# سبق 01 - اے آئی ایجنٹس کا تعارف

**ابتدائیوں کے لیے اے آئی ایجنٹس** کورس کے پہلے سبق میں خوش آمدید!

ایک **اے آئی ایجنٹ** ایسا پروگرام ہے جو ایک بڑی زبان کے ماڈل (LLM) کو اپنے منطق انجن کے طور پر استعمال کرتا ہے اور حقیقی دنیا میں *عمل* کر سکتا ہے — API کال کرنا، ڈیٹا بیس سے معلومات حاصل کرنا، یا کوڈ چلانا — تاکہ صارف کی جانب سے کوئی مقصد حاصل کیا جا سکے۔

اس نوٹ بک میں آپ اپنا پہلا ایجنٹ بنائیں گے: ایک **ٹریول ایجنٹ** جو تعطیلات کی جگہوں کی سفارش کرتا ہے۔ اس راستے میں آپ سیکھیں گے کہ:

1. مائیکروسافٹ ایجنٹ فریم ورک کا استعمال کرتے ہوئے مائیکروسافٹ فاؤنڈری ایجنٹ سروس سے رابطہ قائم کریں۔
2. ایجنٹ کو ایک **ٹول** دیں — ایک سادہ پائتھون فنکشن جسے وہ کال کر سکتا ہے۔
3. ایجنٹ کو چلائیں اور اس کا جواب دیکھیں۔
4. ایجنٹ کے جواب کو ٹوکن بہ ٹوکن سٹریم کریں۔


## ترتیب

اس نوٹ بُک کو چلانے سے پہلے، یقینی بنائیں کہ آپ کے پاس:

1. **ایک Microsoft Foundry پروجیکٹ** جس میں ایک تعینات چیٹ ماڈل ہو (مثلاً `gpt-4.1-mini`)۔
2. **Azure CLI کے ساتھ لاگ ان کیا ہوا** — اپنے ٹرمینل میں `az login` چلائیں۔
3. **ضروری ماحول کی متغیرات سیٹ کیں:**
   - `AZURE_AI_PROJECT_ENDPOINT` — آپ کے Microsoft Foundry پروجیکٹ کا اینڈپوائنٹ۔
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — آپ کے تعینات شدہ ماڈل کا نام۔

نیچے دی گئی سیل وہ Python پیکجز انسٹال کرتی ہے جن کی آپ کو ضرورت ہے۔


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## اپنا پہلا ایجنٹ بنانا  

ایجنٹ کو دو چیزوں کی ضرورت ہوتی ہے:  

- **ہدایات** جو اسے بتاتی ہیں *کہ وہ کون ہے* اور *کیسا برتاؤ کرنا ہے* (ایک سسٹم پرامپٹ)۔  
- **آلات** — Python فنکشنز جو `@tool` سے سجے ہوتے ہیں اور ایجنٹ انہیں معلومات حاصل کرنے یا کاروائی کرنے کے لیے بلا سکتا ہے۔  

نیچے ہم ایک سادہ ٹول تعریف کرتے ہیں جو مشہور تعطیلاتی مقامات کی فہرست واپس کرتا ہے۔ جب صارف سفر کی سفارشات مانگے گا تو ایجنٹ اس ٹول کا استعمال کرے گا۔  


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## سٹریمنگ جوابات  

ایک زیادہ انٹرایکٹو تجربے کے لیے آپ ایجنٹ کے جواب کو **سٹریمنگ** کر سکتے ہیں۔ پورے جواب کا انتظار کرنے کے بجائے، ایجنٹ جتنے جتنا متن تیار ہوتا ہے وہ فراہم کرتا ہے۔ یہ خاص طور پر چیٹ انٹرفیسز میں مفید ہے جہاں آپ حقیقی وقت میں آؤٹ پٹ دکھانا چاہتے ہیں۔  


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## خلاصہ

اس سبق میں آپ نے سیکھا کہ کیسے:

- **ایک پرووائیڈر بنائیں** جو Microsoft Foundry Agent Service کے ساتھ `FoundryChatClient` کے ذریعے جڑتا ہے۔
- **ایک ٹول کی تعریف کریں** `@tool` ڈیکوریٹر کا استعمال کرتے ہوئے تاکہ ایجنٹ آپ کے Python فنکشنز کو کال کر سکے۔
- **ایجنٹ چلائیں** ایک صارف کے پیغام کے ساتھ اور اس کا جواب پرنٹ کریں۔
- **جوابات کو اسٹریم کریں** تاکہ حقیقی وقت میں آؤٹ پٹ حاصل ہو سکے۔

اگلے سبق میں ہم ایجنٹک فریم ورکس کو مزید گہرائی میں دیکھیں گے اور سیکھیں گے کہ ایجنٹس کو زیادہ طاقتور ٹولز اور کثیر مرحلہ استدلال کی صلاحیتیں کیسے دی جاتی ہیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
